[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/risk_portfolios.ipynb)

# Risk and portfolios full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (two kinds of average, the standard deviation, a portfolio of two shares, many shares,
lending and borrowing, Sharpe ratios, beta), **tier 3** stretch problems (combine ideas, spot the trap, run a formula
backwards) and **tier 4**, the case: the shares of Bangalore Cold Chain (a fictional company; our own data) swing
twice as much as the market, and a director reasons that they should earn twice the market's reward for risk. Do
they? Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same returns, risks and portfolios in
plain Python, with `numpy` for the table of covariances and `pandas` for the history of returns and the case's ten
years, so you can run them on your own numbers. And three cells you can move yourself: the weight between two shares
against the portfolio's risk, the number of shares against the risk that never goes away, and the mix of two shares
against the slope of its lending line.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, move it and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Returns,
standard deviations and premiums as percentages: `19` for 19 % (typing the % sign is fine, and so is the rate
itself, `0.19`). Sharpe ratios, betas and correlations to two decimals: `0.25`. A covariance as a decimal to four
places: `0.0300`. You can type the arithmetic too: `sqrt(0.0361)*100`. For several answers, separate them with
commas, in the order the question lists them (the box's label shows the order). You can also put your answer between
the quotes in a cell's last line, `practice("P1.5", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[risk and portfolios cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/risk_portfolios_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, statistics, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # a share's returns in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# a share's returns CHALK · the portfolio SKY · the risk-free rate and the lending line VIOLET · gains MINT · a shortcut
# that misleads CORAL · the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own example · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "ln": math.log, "log": math.log}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…), exp(…), ln(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(dollars?|years?|shares?|points?|percent)\b|[$%]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol, scale=None):
    """Within the tolerance. A percentage may also be typed as the rate itself (0.19 for 19 %: scale 100), and a
    variance in percent squared as the decimal variance (0.00625 for 62.5: scale 10,000)."""
    if isinstance(got, str): return False
    tol = max(tol, 1e-7)
    return abs(got - want) <= tol or (bool(scale) and abs(got * scale - want) <= tol)

def show(v):
    return f"{round(v, 6) + 0.0:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'What history says about risk and return',
     "q": 'A share bought for $25 pays a $0.50 dividend and ends the year at $27. What was its rate of return, in percent?',
     "parts": ['rate of return (%)'],
     "answer": [10.0], "tol": [0.005], "scale": [100], "function": '—',
     "working": '(0.50 + 27 − 25)/25 = 2.50/25 = 10 %'},
    {"id": 'P1.2', "tier": 1, "group": 'What history says about risk and return',
     "q": 'A share returned 14 %, −6 %, 10 % and 2 % in four years. What was its average return, in percent?',
     "parts": ['average return (%)'],
     "answer": [5.0], "tol": [0.005], "scale": [100], "function": 'AVERAGE',
     "working": '(14 − 6 + 10 + 2)/4 = 20/4 = 5 %'},
    {"id": 'P1.3', "tier": 1, "group": 'What history says about risk and return',
     "q": 'From 1928 to 2025, US stocks averaged 11.86 % a year and Treasury bills 3.42 %. What was the risk premium of stocks over bills, in percentage points?',
     "parts": ['the risk premium (% points)'],
     "answer": [8.44], "tol": [0.005], "scale": [100], "function": '—',
     "working": '11.86 − 3.42 = 8.44 points'},
    {"id": 'P1.4', "tier": 1, "group": 'What history says about risk and return',
     "q": 'A fund gains 20 % one year and loses 20 % the next. What is its compound average return, in percent?',
     "parts": ['compound average return (%)'],
     "answer": [-2.0204102886728803], "tol": [0.005], "scale": [100], "function": 'GEOMEAN',
     "working": '1.2 × 0.8 = 0.96; 0.96^(1/2) − 1 = -2.02 % (the plain average is 0 %)'},
    {"id": 'P1.5', "tier": 1, "group": 'Measuring risk',
     "q": 'Four yearly returns: 4 %, 12 %, −2 % and 10 %. What is their standard deviation, in percent (divide by n − 1)?',
     "parts": ['standard deviation (%)'],
     "answer": [6.324555320336758], "tol": [0.005], "scale": [100], "function": 'STDEV.S',
     "working": 'mean 6; deviations −2, 6, −8, 4; squares 4 + 36 + 64 + 16 = 120; 120/3 = 40; √40 = 6.32 %'},
    {"id": 'P1.6', "tier": 1, "group": 'Diversification',
     "q": "A portfolio holds 30 % in a share expected to return 12 % and 70 % in one expected to return 6 %. What is the portfolio's expected return, in percent?",
     "parts": ['expected return (%)'],
     "answer": [7.799999999999999], "tol": [0.005], "scale": [100], "function": 'SUMPRODUCT',
     "working": '0.3 × 12 + 0.7 × 6 = 3.6 + 4.2 = 7.8 %'},
    {"id": 'P1.7', "tier": 1, "group": 'Diversification',
     "q": 'Two shares have standard deviations of 20 % and 30 % and a correlation of 0.5. What is their covariance, as a decimal to four places?',
     "parts": ['covariance (a decimal, four places)'],
     "answer": [0.03], "tol": [5e-05], "scale": [None], "function": '—',
     "working": 'covariance = correlation × σ1 × σ2 = 0.5 × 0.20 × 0.30 = 0.0300'},
    {"id": 'P1.8', "tier": 1, "group": 'Efficient portfolios, lending and borrowing',
     "q": 'A portfolio expects 10 % with a standard deviation of 24 %; the risk-free rate is 4 %. What is its Sharpe ratio?',
     "parts": ['Sharpe ratio'],
     "answer": [0.25000000000000006], "tol": [0.005], "scale": [None], "function": '—',
     "working": '(10 − 4)/24 = 0.25'},
    {"id": 'P1.9', "tier": 1, "group": 'Efficient portfolios, lending and borrowing',
     "q": "You put half your money in the market portfolio (expected return 9 %, standard deviation 20 %) and lend half at 4 %. Give the mix's expected return and standard deviation, in percent.",
     "parts": ['expected return (%)', 'standard deviation (%)'],
     "answer": [6.5, 10.0], "tol": [0.005, 0.005], "scale": [100, 100], "function": '—',
     "working": 'expected return 0.5 × 9 + 0.5 × 4 = 6.5 %; standard deviation 0.5 × 20 = 10 %'},
    {"id": 'P1.10', "tier": 1, "group": 'Market risk and beta',
     "q": "A share's standard deviation is 32 % and its correlation with the market is 0.5. The market's standard deviation is 20 %. What is the share's beta?",
     "parts": ['beta'],
     "answer": [0.7999999999999999], "tol": [0.005], "scale": [None], "function": '—',
     "working": 'β = ρ × σ/σm = 0.5 × 32/20 = 0.8'},
    {"id": 'P2.1', "tier": 2, "group": 'What history says about risk and return',
     "q": 'Treasury bills yield 3.5 % today. If investors expect the market to pay a premium of 6 points over bills, what return do they expect from the market, in percent?',
     "parts": ["the market's expected return (%)"],
     "answer": [9.5], "tol": [0.005], "scale": [100], "function": '—',
     "working": "expected market return = today's risk-free rate + the premium = 3.5 + 6 = 9.5 %"},
    {"id": 'P2.2', "tier": 2, "group": 'What history says about risk and return',
     "q": 'A share returns +30 %, −20 % and +25 % in three years. Give its average return and its compound average return, in percent.',
     "parts": ['average return (%)', 'compound average return (%)'],
     "answer": [11.666666666666666, 9.139288306110593], "tol": [0.005, 0.005], "scale": [100, 100], "function": 'AVERAGE, GEOMEAN',
     "working": 'average (30 − 20 + 25)/3 = 11.67 %; growth 1.3 × 0.8 × 1.25 = 1.30, so compound 1.30^(1/3) − 1 = 9.14 %. =AVERAGE(...) and =GEOMEAN(1.3, 0.8, 1.25)-1'},
    {"id": 'P2.3', "tier": 2, "group": 'Measuring risk',
     "q": 'Five yearly returns: 15 %, −5 %, 10 %, 0 % and 5 %. Give the mean (percent), the variance (in percent squared, dividing by n − 1) and the standard deviation (percent).',
     "parts": ['mean (%)', 'variance (percent squared)', 'standard deviation (%)'],
     "answer": [5.0, 62.5, 7.905694150420948], "tol": [0.005, 0.005, 0.005], "scale": [100, 10000, 100], "function": 'AVERAGE, VAR.S, STDEV.S',
     "working": 'mean 5; deviations 10, −10, 5, −5, 0; squares sum to 250; 250/4 = 62.5; √62.5 = 7.91 %. =AVERAGE, =VAR.S, =STDEV.S'},
    {"id": 'P2.4', "tier": 2, "group": 'Measuring risk',
     "q": 'US stocks averaged 11.86 % a year with a standard deviation of 19.40 % (1928–2025). What range lies within one standard deviation of the average? Give the low and the high end, in percent.',
     "parts": ['the low end (%)', 'the high end (%)'],
     "answer": [-7.539999999999999, 31.259999999999998], "tol": [0.005, 0.005], "scale": [100, 100], "function": '—',
     "working": '11.86 − 19.40 = −7.54 % and 11.86 + 19.40 = 31.26 % (65 of the 98 years fell inside it)'},
    {"id": 'P2.5', "tier": 2, "group": 'Diversification',
     "q": "Half your money goes into a retailer (expected return 7 %, standard deviation 16 %) and half into a bank (11 %, 30 %). Their correlation is 0.3. Give the portfolio's expected return and standard deviation, in percent.",
     "parts": ['expected return (%)', 'standard deviation (%)'],
     "answer": [9.0, 19.0], "tol": [0.005, 0.005], "scale": [100, 100], "function": 'SUMPRODUCT, SQRT',
     "working": 'expected return 0.5 × 7 + 0.5 × 11 = 9 %; variance 0.25 × 0.0256 + 0.25 × 0.09 + 2 × 0.25 × 0.3 × 0.16 × 0.30 = 0.0064 + 0.0225 + 0.0072 = 0.0361; √0.0361 = 19 % (the average of the two is 23 %)'},
    {"id": 'P2.6', "tier": 2, "group": 'Diversification',
     "q": 'Same two shares, but 70 % in the retailer and 30 % in the bank. Give the expected return and the standard deviation, in percent.',
     "parts": ['expected return (%)', 'standard deviation (%)'],
     "answer": [8.200000000000001, 16.33768649472746], "tol": [0.005, 0.005], "scale": [100, 100], "function": 'SUMPRODUCT, SQRT',
     "working": 'expected return 0.7 × 7 + 0.3 × 11 = 8.2 %; variance 0.49 × 0.0256 + 0.09 × 0.09 + 2 × 0.21 × 0.0144 = 0.026692; standard deviation 16.34 %'},
    {"id": 'P2.7', "tier": 2, "group": 'Diversification',
     "q": "A portfolio holds 25 shares in equal amounts. Each has a standard deviation of 50 %, and every pair has a covariance of 0.04. Give the portfolio's standard deviation, and the floor it can never go below, in percent.",
     "parts": ['standard deviation (%)', 'the floor (%)'],
     "answer": [22.0, 20.0], "tol": [0.005, 0.005], "scale": [100, 100], "function": 'SQRT',
     "working": 'variance = 0.25/25 + (1 − 1/25) × 0.04 = 0.01 + 0.0384 = 0.0484; √ = 22 %; floor = √0.04 = 20 %, the average covariance'},
    {"id": 'P2.8', "tier": 2, "group": 'Efficient portfolios, lending and borrowing',
     "q": 'You put 60 % in the market portfolio (9 %, 20 %) and lend 40 % at 4 %. Give the expected return and the standard deviation, in percent.',
     "parts": ['expected return (%)', 'standard deviation (%)'],
     "answer": [6.999999999999999, 12.0], "tol": [0.005, 0.005], "scale": [100, 100], "function": '—',
     "working": 'expected return 4 + 0.6 × (9 − 4) = 7 %; standard deviation 0.6 × 20 = 12 %'},
    {"id": 'P2.9', "tier": 2, "group": 'Efficient portfolios, lending and borrowing',
     "q": 'Three funds: X expects 8 % with a standard deviation of 16 %; Y 10 % with 30 %; Z 7 % with 10 %. The risk-free rate is 4 %. Give the three Sharpe ratios (X, Y, Z). Which fund should be mixed with lending or borrowing?',
     "parts": ['Sharpe ratio of X', 'Sharpe ratio of Y', 'Sharpe ratio of Z'],
     "answer": [0.25, 0.2, 0.30000000000000004], "tol": [0.005, 0.005, 0.005], "scale": [None, None, None], "function": '—',
     "working": 'X (8 − 4)/16 = 0.25; Y (10 − 4)/30 = 0.20; Z (7 − 4)/10 = 0.30: Z, the steepest line'},
    {"id": 'P2.10', "tier": 2, "group": 'Market risk and beta',
     "q": 'A share has a standard deviation of 25 % and a correlation of 0.6 with the market (standard deviation 20 %). Give its beta, its market risk and its specific risk (the last two as standard deviations, in percent).',
     "parts": ['beta', 'market risk (%)', 'specific risk (%)'],
     "answer": [0.7499999999999999, 15.0, 20.0], "tol": [0.005, 0.005, 0.005], "scale": [None, 100, 100], "function": 'SQRT',
     "working": 'β = 0.6 × 25/20 = 0.75; market part 0.75 × 20 = 15 %; specific part √(25² − 15²) = √400 = 20 %'},
    {"id": 'P3.1', "tier": 3, "group": 'What history says about risk and return',
     "q": 'A share returned +30 % and then −20 %. What must it return in the third year for its compound average over the three years to be exactly 10 % a year? In percent.',
     "parts": ["the third year's return (%)"],
     "answer": [27.980769230769265], "tol": [0.005], "scale": [100], "function": '—',
     "working": 'it must end at 1.1³ = 1.331; it stands at 1.3 × 0.8 = 1.04; 1.331/1.04 − 1 = 27.98 %'},
    {"id": 'P3.2', "tier": 3, "group": 'Measuring risk',
     "q": 'Ten yearly returns have a standard deviation of 15 %. What do their squared deviations add up to (in percent squared)? An eleventh year then lands exactly on the average: what is the new standard deviation, in percent?',
     "parts": ["the squared deviations' sum (percent squared)", 'the new standard deviation (%)'],
     "answer": [2025.0, 14.230249470757709], "tol": [0.005, 0.005], "scale": [10000, 100], "function": 'SQRT',
     "working": '(n − 1) × 15² = 9 × 225 = 2,025; the new year adds nothing to the sum but one to the count: √(2,025/10) = 14.23 %'},
    {"id": 'P3.3', "tier": 3, "group": 'Diversification',
     "q": 'A portfolio is half in each of two shares with standard deviations of 20 % and 40 %. Its own standard deviation is 26 %. What is the correlation between the two shares?',
     "parts": ['correlation'],
     "answer": [0.44], "tol": [0.005], "scale": [None], "function": '—',
     "working": '0.26² = 0.0676 = 0.25 × 0.04 + 0.25 × 0.16 + 2 × 0.25 × ρ × 0.08 = 0.05 + 0.04ρ, so ρ = 0.0176/0.04 = 0.44'},
    {"id": 'P3.4', "tier": 3, "group": 'Diversification',
     "q": "Two shares have standard deviations of 20 % and 30 %. With 60 % in the first and 40 % in the second, what is the portfolio's standard deviation if their correlation is +1? And if it is −1? In percent.",
     "parts": ['standard deviation at +1 (%)', 'standard deviation at −1 (%)'],
     "answer": [24.0, 0.0], "tol": [0.005, 0.005], "scale": [100, 100], "function": 'SQRT',
     "working": 'ρ = +1: 0.6 × 20 + 0.4 × 30 = 24 %, the plain average, no gain. ρ = −1: 0.6 × 20 − 0.4 × 30 = 0: the two risks cancel exactly at these weights'},
    {"id": 'P3.5', "tier": 3, "group": 'Diversification',
     "q": "Shares each have a standard deviation of 50 %, and every pair a covariance of 0.04. How many shares, in equal amounts, bring the portfolio's standard deviation down to 21 % or less?",
     "parts": ['number of shares'],
     "answer": [52.0], "tol": [0], "scale": [None], "function": '—',
     "working": '0.21² = 0.0441 = 0.04 + (0.25 − 0.04)/n, so n = 0.21/0.0041 = 51.2: 52 shares'},
    {"id": 'P3.6', "tier": 3, "group": 'Efficient portfolios, lending and borrowing',
     "q": 'You want an expected return of 11 % using only the market portfolio (9 %, 20 %) and lending or borrowing at 4 %. What weight do you put in the market (as a number, 1 = all of your money), and what standard deviation do you accept, in percent?',
     "parts": ['weight in the market (1 = all your money)', 'standard deviation (%)'],
     "answer": [1.4000000000000004, 28.000000000000007], "tol": [0.005, 0.005], "scale": [None, 100], "function": '—',
     "working": 'weight = (11 − 4)/(9 − 4) = 1.4: borrow 40 % of your own money; standard deviation 1.4 × 20 = 28 %'},
    {"id": 'P3.7', "tier": 3, "group": 'Efficient portfolios, lending and borrowing',
     "q": 'The bakery expects 7 % with a standard deviation of 20 %; the software firm 10 % with 30 %; correlation 0.25. Which weight in the bakery gives the lowest standard deviation (try steps of 5 %), and what is it? In percent.',
     "parts": ['weight in the bakery (%)', 'the lowest standard deviation (%)'],
     "answer": [75.0, 18.371173070873837], "tol": [0.005, 0.005], "scale": [100, 100], "function": 'SQRT',
     "working": "75 % in the bakery: variance 0.5625 × 0.04 + 0.0625 × 0.09 + 2 × 0.1875 × 0.015 = 0.03375, standard deviation 18.37 %, below the bakery's own 20 %"},
    {"id": 'P3.8', "tier": 3, "group": 'Market risk and beta',
     "q": "A share has a beta of 0.8 and a standard deviation of 20 %, the same as the market's. What is its correlation with the market, and how large is its specific risk (a standard deviation, in percent)?",
     "parts": ['correlation with the market', 'specific risk (%)'],
     "answer": [0.8000000000000002, 11.999999999999998], "tol": [0.005, 0.005], "scale": [None, 100], "function": 'SQRT',
     "working": 'ρ = β × σm/σ = 0.8 × 20/20 = 0.8; market part 0.8 × 20 = 16 %; specific part √(20² − 16²) = 12 %'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'From the ten years of returns, find the average return and the standard deviation of the market and of the shares (four answers, in percent: market average, share average, market standard deviation, share standard deviation).',
     "parts": ["the market's average (%)", "the shares' average (%)", "the market's standard deviation (%)", "the shares' standard deviation (%)"],
     "answer": [9.000000000000002, 10.0, 20.000000000000004, 40.0], "tol": [0.005, 0.005, 0.005, 0.005], "scale": [100, 100, 100, 100], "function": 'AVERAGE, STDEV.S',
     "working": '=AVERAGE and =STDEV.S of each column: 9 %, 10 %, 20 %, 40 % (the data are built so that these come out round)'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Find the covariance between the shares and the market (a decimal, four places) and their correlation.',
     "parts": ['covariance (a decimal, four places)', 'correlation'],
     "answer": [0.04799999999999999, 0.5999999999999998], "tol": [5e-05, 0.005], "scale": [None, None], "function": 'COVARIANCE.S, CORREL',
     "working": '=COVARIANCE.S(shares, market) = 0.0480; =CORREL(shares, market) = 0.60 = 0.048/(0.40 × 0.20)'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Find the shares' beta two ways: from the covariance and the market's variance, and from the correlation and the two standard deviations.",
     "parts": ['beta from the covariance', 'beta from the correlation'],
     "answer": [1.1999999999999993, 1.1999999999999993], "tol": [0.005, 0.005], "scale": [None, None], "function": 'COVARIANCE.S, VAR.S',
     "working": '0.048/0.04 = 1.2; 0.6 × 40/20 = 1.2'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "A director says the shares are twice as risky as the market, so shareholders need twice the market's 5-point premium over the 4 % risk-free rate. What return does that give, and what return does a mix of the market and borrowing with the same market risk expect? In percent.",
     "parts": ["the director's shortcut (%)", 'the market-and-borrowing mix (%)'],
     "answer": [13.999999999999998, 9.999999999999995], "tol": [0.005, 0.005], "scale": [100, 100], "function": '—',
     "working": 'the shortcut: 4 + 2 × 5 = 14 %; with beta: 1.2 in the market, 0.2 borrowed: 4 + 1.2 × 5 = 10 %'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Split the shares' 40 % standard deviation into the part that moves with the market and the specific part (in percent). What share of the variance is market risk, in percent?",
     "parts": ['market part (%)', 'specific part (%)', "market risk's share of the variance (%)"],
     "answer": [23.99999999999999, 32.000000000000014, 35.999999999999964], "tol": [0.005, 0.005, 0.005], "scale": [100, 100, 100], "function": 'SQRT',
     "working": 'market part 1.2 × 20 = 24 %; specific part √(40² − 24²) = 32 %; 24²/40² = 36 % of the variance'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "An investor holds the market portfolio. What is her portfolio's standard deviation if she moves 10 % of it into the shares? And 25 %? In percent.",
     "parts": ['with 10 % in the shares (%)', 'with 25 % in the shares (%)'],
     "answer": [20.649455198624494, 22.472205054244235], "tol": [0.005, 0.005], "scale": [100, 100], "function": 'SQRT',
     "working": '10 %: √(0.81 × 0.04 + 0.01 × 0.16 + 2 × 0.9 × 0.1 × 0.048) = √0.04264 = 20.65 %; 25 %: √0.0505 = 22.47 %'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Imagine a portfolio of 16 shares like these, in equal amounts: the same beta, and specific risks that have nothing to do with each other. What is its standard deviation, and what floor does it approach with more shares? In percent.',
     "parts": ['16 companies like it (%)', 'the floor (%)'],
     "answer": [25.29822128134703, 23.99999999999999], "tol": [0.005, 0.005], "scale": [100, 100], "function": 'SQRT',
     "working": '√(0.24² + 0.32²/16) = √(0.0576 + 0.0064) = 25.30 %; the floor is the market part, 24 %'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Write the recommendation in two sentences, with the numbers that anchor it. Give the return shareholders ask of the shares, in percent.',
     "parts": ['the return shareholders ask (%)'],
     "answer": [9.999999999999995], "tol": [0.005], "scale": [100], "function": '—',
     "working": 'Shareholders who hold diversified portfolios ask about 10 % of the shares, not 14 %: only 24 of the 40 points of risk move with the market (beta 1.2); the other 32 wash out in a portfolio and earn no reward. The company should not diversify to calm its share price: its investors already have'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t, s) for g, w, t, s in zip(got, q["answer"], q["tol"], q["scale"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(float(g), w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't; a percentage may be typed as itself or as the rate
assert right(read_part("sqrt(40)"), answers("P1.5")[0], 0.005) and not right(read_part("6.3"), answers("P1.5")[0], 0.005)
assert right(read_part("19%"), 19.0, 0.005, 100) and right(read_part("0.19"), 19.0, 0.005, 100)
assert not right(read_part("0.19"), 19.0, 0.005) and not right(read_part("nineteen"), 19.0, 0.005, 100)
assert right(read_part("0.00625"), 62.5, 0.005, 10000) and not right(read_part("0.625"), 62.5, 0.005, 10000)
assert show(-2.0204102886728803) == "−2.02041" and show(2.6e-7) == "0" and show(52) == "52"

## Tier 1 · Warm-up

One idea, one step, small numbers: a rate of return, an average, the risk premium, a compound average, a standard
deviation, a portfolio's expected return, a covariance from a correlation, a Sharpe ratio, a mix with lending, and a
beta.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Python's own `statistics` module has the average and the standard deviation: `statistics.mean` is a spreadsheet's
AVERAGE, and `statistics.stdev` is STDEV.S (it divides by one fewer than the count). Everything else from the course
is one line. The cell below writes those lines once (the comment beside each names the spreadsheet function that does
the same job), then computes every warm-up answer and checks it against the practice set.

In [ ]:
def rate_of_return(paid, worth_later, dividend=0.0): return (dividend + worth_later - paid) / paid
def compound_average(returns):                                     # GEOMEAN(1 + each return) − 1
    return math.prod(1 + r for r in returns) ** (1 / len(returns)) - 1
def portfolio_return(weights, returns):                            # SUMPRODUCT(weights, returns)
    return sum(w * r for w, r in zip(weights, returns))
def sharpe(expected, sd, risk_free): return (expected - risk_free) / sd       # reward for each unit of risk
def mix(w, expected, sd, risk_free):                               # w in a risky portfolio, the rest lent
    return risk_free + w * (expected - risk_free), abs(w) * sd     # (w above 1: the extra is borrowed)
def beta(correlation, sd, sd_market): return correlation * sd / sd_market

one_year = rate_of_return(25, 27, dividend=0.50)                   # P1.1
average = statistics.mean([14, -6, 10, 2])                         # P1.2: AVERAGE
premium = 11.86 - 3.42                                             # P1.3: stocks over bills, in points
up_down = compound_average([0.20, -0.20])                          # P1.4: up 20 %, then down 20 %
spread = statistics.stdev([4, 12, -2, 10])                         # P1.5: STDEV.S
expected = portfolio_return([0.3, 0.7], [12, 6])                   # P1.6
covariance_17 = 0.5 * 0.20 * 0.30                                  # P1.7: correlation × σ1 × σ2
slope = sharpe(10, 24, 4)                                          # P1.8
half_lent = mix(0.5, 9, 20, 4)                                     # P1.9: half in the market, half lent
beta_110 = beta(0.5, 32, 20)                                       # P1.10

print(f"return {one_year:.1%}   average {average:g} %   premium {premium:.2f} points   compound average {up_down:.2%}")
print(f"standard deviation {spread:.2f} %   portfolio {expected:.1f} %   covariance {covariance_17:.4f}   "
      f"Sharpe ratio {slope:.2f}")
print(f"half lent: {half_lent[0]:g} % expected, {half_lent[1]:g} % standard deviation   beta {beta_110:.2f}")
for k, v in (("P1.1", [100 * one_year]), ("P1.2", [average]), ("P1.3", [premium]), ("P1.4", [100 * up_down]),
             ("P1.5", [spread]), ("P1.6", [expected]), ("P1.7", [covariance_17]), ("P1.8", [slope]),
             ("P1.9", half_lent), ("P1.10", [beta_110])):
    assert agrees(k, v), k

### Two shares, five years

The course's two shares: a bakery chain and a software firm, five yearly returns each. The bakery averaged 7 % a
year with a standard deviation of 20 %; the software firm 10 % with 30 %. Dividing by the full count instead
(`statistics.pstdev`, a spreadsheet's STDEV.P) gives 17.9 % for the bakery, which is too low for a sample of five.
And the bakery's compound average is only 5.47 %: the swings cost growth.

In [ ]:
bakery = [0.12, -0.18, -0.08, 0.17, 0.32]
software = [0.55, 0.05, -0.25, 0.20, -0.05]

for name, share in (("bakery", bakery), ("software", software)):
    print(f"{name:9s} average {statistics.mean(share):.2%}   variance {statistics.variance(share):.4f}   "
          f"standard deviation {statistics.stdev(share):.2%}   compound average {compound_average(share):.2%}")
print(f"dividing by 5 instead of 4: {statistics.pstdev(bakery):.1%}")
assert round(statistics.mean(bakery), 9) == 0.07 and round(statistics.mean(software), 9) == 0.10       # the video's numbers
assert round(statistics.variance(bakery), 9) == 0.04 and round(statistics.stdev(bakery), 9) == 0.20
assert round(statistics.stdev(software), 9) == 0.30 and round(100 * statistics.pstdev(bakery), 1) == 17.9
assert round(100 * compound_average(bakery), 2) == 5.47

### The history, as a table

Ninety-eight years of yearly returns in the United States, 1928 to 2025: stocks (the S&P 500 with dividends),
Treasury bills and ten-year Treasury bonds. This is real, public data (its source is under the picture and at the
end of this notebook); the cell types it in, so nothing is downloaded. `pandas` takes the average and the standard
deviation of every column at once; its `.std()` divides by one fewer than the count, like STDEV.S.

In [ ]:
# The yearly returns, in percent, as the source prints them.
# @HISTORY
YEARS = list(range(1928, 2026))
STOCKS = [
    43.81, -8.30, -25.12, -43.84, -8.64, 49.98, -1.19, 46.74, 31.94, -35.34,
    29.28, -1.10, -10.67, -12.77, 19.17, 25.06, 19.03, 35.82, -8.43, 5.20,
    5.70, 18.30, 30.81, 23.68, 18.15, -1.21, 52.56, 32.60, 7.44, -10.46,
    43.72, 12.06, 0.34, 26.64, -8.81, 22.61, 16.42, 12.40, -9.97, 23.80,
    10.81, -8.24, 3.56, 14.22, 18.76, -14.31, -25.90, 37.00, 23.83, -6.98,
    6.51, 18.52, 31.74, -4.70, 20.42, 22.34, 6.15, 31.24, 18.49, 5.81,
    16.54, 31.48, -3.06, 30.23, 7.49, 9.97, 1.33, 37.20, 22.68, 33.10,
    28.34, 20.89, -9.03, -11.85, -21.97, 28.36, 10.74, 4.83, 15.61, 5.48,
    -36.55, 25.94, 14.82, 2.10, 15.89, 32.15, 13.52, 1.38, 11.77, 21.61,
    -4.23, 31.21, 18.02, 28.47, -18.04, 26.06, 24.88, 17.78,
]
BILLS = [
    3.08, 3.16, 4.55, 2.31, 1.07, 0.96, 0.28, 0.17, 0.17, 0.28,
    0.07, 0.05, 0.04, 0.13, 0.34, 0.38, 0.38, 0.38, 0.38, 0.60,
    1.05, 1.12, 1.20, 1.52, 1.72, 1.89, 0.94, 1.72, 2.62, 3.22,
    1.77, 3.39, 2.87, 2.35, 2.77, 3.16, 3.55, 3.95, 4.86, 4.29,
    5.34, 6.67, 6.39, 4.33, 4.06, 7.04, 7.85, 5.79, 4.98, 5.26,
    7.18, 10.05, 11.39, 14.04, 11.09, 8.95, 9.92, 7.72, 6.15, 5.96,
    6.89, 8.39, 7.75, 5.54, 3.51, 3.07, 4.37, 5.66, 5.15, 5.20,
    4.91, 4.78, 6.00, 3.48, 1.64, 1.03, 1.40, 3.22, 4.85, 4.48,
    1.40, 0.15, 0.14, 0.05, 0.09, 0.06, 0.03, 0.05, 0.32, 0.95,
    1.97, 2.11, 0.36, 0.04, 2.09, 5.28, 5.18, 4.21,
]
BONDS = [
    0.84, 4.20, 4.54, -2.56, 8.79, 1.86, 7.96, 4.47, 5.02, 1.38,
    4.21, 4.41, 5.40, -2.02, 2.29, 2.49, 2.58, 3.80, 3.13, 0.92,
    1.95, 4.66, 0.43, -0.30, 2.27, 4.14, 3.29, -1.34, -2.26, 6.80,
    -2.10, -2.65, 11.64, 2.06, 5.69, 1.68, 3.73, 0.72, 2.91, -1.58,
    3.27, -5.01, 16.75, 9.79, 2.82, 3.66, 1.99, 3.61, 15.98, 1.29,
    -0.78, 0.67, -2.99, 8.20, 32.81, 3.20, 13.73, 25.71, 24.28, -4.96,
    8.22, 17.69, 6.24, 15.00, 9.36, 14.21, -8.04, 23.48, 1.43, 9.94,
    14.92, -8.25, 16.66, 5.57, 15.12, 0.38, 4.49, 2.87, 1.96, 10.21,
    20.10, -11.12, 8.46, 16.04, 2.97, -9.10, 10.75, 1.28, 0.69, 2.80,
    -0.02, 9.64, 11.33, -4.42, -17.83, 3.88, -1.64, 7.80,
]
# @END HISTORY
history = pd.DataFrame({"year": YEARS, "stocks": STOCKS, "bills": BILLS, "bonds": BONDS}).set_index("year") / 100

summary = pd.DataFrame({"average": history.mean(), "standard deviation": history.std(),
                        "compound average": (1 + history).prod() ** (1 / len(history)) - 1,
                        "$1 grew to": (1 + history).prod(), "years with a loss": (history < 0).sum()})
risk_premium = history["stocks"].mean() - history["bills"].mean()
print(f"{len(history)} years, {history.index[0]} to {history.index[-1]}")
print(summary.assign(**{c: (100 * summary[c]).round(2) for c in summary.columns[:3]}).round(2).to_string())
print(f"\nrisk premium of stocks over bills: {100 * risk_premium:.2f} points a year")
m_, s_ = history["stocks"].mean(), history["stocks"].std()
inside = int(((history["stocks"] - m_).abs() <= s_).sum())
print(f"stocks landed within one standard deviation of their average ({m_ - s_:.1%} to {m_ + s_:.1%}) in {inside} of "
      f"{len(history)} years")
assert len(history) == 98 and [round(100 * v, 2) for v in summary["average"]] == [11.86, 3.42, 4.82]   # the video's numbers
assert [round(100 * v, 1) for v in summary["standard deviation"]] == [19.4, 3.0, 7.9] and inside == 65
assert round(100 * risk_premium, 2) == 8.44 and list(summary["years with a loss"]) == [26, 0, 20]
assert round(100 * history["stocks"].std(ddof=0), 1) == 19.3 and round(summary["$1 grew to"]["stocks"]) == 11576

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6))
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
ax.bar(history.index, 100 * history["stocks"], width=0.8, color=[MINT if v >= 0 else CORAL for v in history["stocks"]])
ax.axhspan(100 * (m_ - s_), 100 * (m_ + s_), color=GOLD, alpha=0.12)
ax.axhline(100 * m_, color=GOLD, lw=1.8)
ax.text(1929, 64, f"the line: the average, {100 * m_:.1f} % · the band: one standard deviation either side", color=GOLD,
        fontsize=12, va="center")
ax.set_xlim(1927, 2026); ax.set_ylim(-50, 72); ax.set_ylabel("return in the year (%)")
fig.suptitle(f"US stocks, 1928–2025: a standard deviation of {100 * s_:.1f} %; {inside} of 98 years inside the band",
             x=0.08, ha="left", fontsize=15, color=GOLD)
footnote(fig, "A. Damodaran, NYU Stern · annual returns 1928–2025 · retrieved 2026-11-11 · computed"); plt.show()

## Tier 2 · Core

The module's standard problems, several steps each: an expected market return, the two kinds of average, a variance
and a standard deviation, a one-standard-deviation range, two portfolios of a retailer and a bank, a portfolio of
twenty-five shares, a mix of the market and lending, three funds' Sharpe ratios, and a share's risk split into its
market part and its specific part.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python: the portfolio in three lines

Half in the bakery, half in the software firm. `np.cov` builds the table of covariances from the two columns of
returns: each share's variance on the diagonal, the covariance between them in the other two boxes (it divides by
one fewer than the count, like VAR.S and COVARIANCE.S). The portfolio's variance is the weights times that table
times the weights: every box, multiplied by its two weights and added up. The `@` sign multiplies arrays that way.

In [ ]:
weights = np.array([0.5, 0.5])
covariance = np.cov(bakery, software)
variance = weights @ covariance @ weights
print(f"standard deviation = {np.sqrt(variance):.1%}")

In [ ]:
# The table itself, the correlation, and the same variance from the three-piece formula.
def two_share_sd(w, sd1, sd2, correlation):
    """w in the first share, 1 − w in the second: SQRT(w²σ1² + (1 − w)²σ2² + 2 w (1 − w) ρ σ1 σ2)."""
    v = w ** 2 * sd1 ** 2 + (1 - w) ** 2 * sd2 ** 2 + 2 * w * (1 - w) * correlation * sd1 * sd2
    return np.sqrt(np.maximum(v, 0.0))

MEAN_B, MEAN_S = np.mean(bakery), np.mean(software)                # 7 % · 10 %
SD_B, SD_S = np.sqrt(covariance[0, 0]), np.sqrt(covariance[1, 1])  # 20 % · 30 %
RHO = np.corrcoef(bakery, software)[0, 1]                          # CORREL: 0.25
print(np.round(covariance, 4))
print(f"covariance {covariance[0, 1]:.4f} (COVARIANCE.S)   correlation {RHO:.2f} (CORREL) = "
      f"{covariance[0, 1]:.4f} / ({SD_B:.2f} × {SD_S:.2f})")
print(f"expected return {weights @ np.array([MEAN_B, MEAN_S]):.2%}   three pieces: "
      f"{0.25 * SD_B ** 2:.4f} + {0.25 * SD_S ** 2:.4f} + {2 * 0.25 * covariance[0, 1]:.4f} = {variance:.4f}")
own = 0.5 * np.array(bakery) + 0.5 * np.array(software)            # the portfolio's own five yearly returns
print(f"the portfolio's own yearly returns {np.round(100 * own, 1)} have a standard deviation of {np.std(own, ddof=1):.1%}")
assert round(float(variance), 9) == 0.04 and round(float(covariance[0, 1]), 9) == 0.015 and round(float(RHO), 9) == 0.25
assert abs(two_share_sd(0.5, SD_B, SD_S, RHO) - np.sqrt(variance)) < 1e-12 and abs(np.std(own, ddof=1) - 0.2) < 1e-12
assert [round(100 * float(two_share_sd(0.5, SD_B, SD_S, c)), 1) for c in (1, 0, -1)] == [25.0, 18.0, 5.0]   # the edge tests

In [ ]:
# The core answers, each checked against the practice set.
def equal_weight_sd(n, average_variance, average_covariance):
    """n shares in equal amounts: SQRT(average variance ÷ n + (1 − 1 ÷ n) × average covariance)."""
    return np.sqrt(average_variance / n + (1 - 1 / n) * average_covariance)

market_return = 3.5 + 6                                            # P2.1: today's risk-free rate + the premium
three = [0.30, -0.20, 0.25]                                        # P2.2
five = [15, -5, 10, 0, 5]                                          # P2.3: in percent, so the variance is in percent squared
low, high = 11.86 - 19.40, 11.86 + 19.40                           # P2.4
retailer_bank = two_share_sd(0.5, 0.16, 0.30, 0.3)                 # P2.5
tilted = two_share_sd(0.7, 0.16, 0.30, 0.3)                        # P2.6
twenty_five, floor = equal_weight_sd(25, 0.50 ** 2, 0.04), math.sqrt(0.04)    # P2.7
market_lent = mix(0.6, 9, 20, 4)                                   # P2.8
funds = {"X": (8, 16), "Y": (10, 30), "Z": (7, 10)}                # P2.9
slopes = [sharpe(e, s, 4) for e, s in funds.values()]
beta_210 = beta(0.6, 25, 20)                                       # P2.10
market_part = beta_210 * 20
specific_part = math.sqrt(25 ** 2 - market_part ** 2)

print(f"market {market_return} %   average {statistics.mean(three):.2%}, compound {compound_average(three):.2%}   "
      f"variance {statistics.variance(five)} %², standard deviation {statistics.stdev(five):.2f} %")
print(f"range {low:.2f} % to {high:.2f} %   retailer and bank: {retailer_bank:.2%} at half each, {tilted:.2%} at 70/30")
print(f"25 shares: {twenty_five:.1%}, floor {floor:.0%}   60 % in the market: {market_lent[0]:g} % at {market_lent[1]:g} %")
print("Sharpe ratios: " + "   ".join(f"{k} {v:.2f}" for k, v in zip(funds, slopes))
      + f"   beta {beta_210:.2f}: market part {market_part:g} %, specific part {specific_part:g} %")
for k, v in (("P2.1", [market_return]), ("P2.2", [100 * statistics.mean(three), 100 * compound_average(three)]),
             ("P2.3", [statistics.mean(five), statistics.variance(five), statistics.stdev(five)]), ("P2.4", [low, high]),
             ("P2.5", [portfolio_return([0.5, 0.5], [7, 11]), 100 * retailer_bank]),
             ("P2.6", [portfolio_return([0.7, 0.3], [7, 11]), 100 * tilted]), ("P2.7", [100 * twenty_five, 100 * floor]),
             ("P2.8", market_lent), ("P2.9", slopes), ("P2.10", [beta_210, market_part, specific_part])):
    assert agrees(k, v), k

### Slide the weight, and the correlation

The SKY curve is the portfolio's risk for every split between the bakery and the software firm; the dashed line is
the plain average of the two risks, which is what you'd get if spreading did nothing. At the shares' own correlation
of 0.25, half in each has a standard deviation of 20 %, not 25 %, and 75 % in the bakery is the safest mix of all:
18.4 %, below either share alone. Now slide the correlation. At +1 the curve lies on the dashed line: no gain. At −1
it touches zero when 60 % is in the bakery: the two risks cancel.

In [ ]:
def weight_picture(bakery_pct=50.0, correlation=0.25):
    w, ws = bakery_pct / 100, np.linspace(0, 1, 201)
    sd = float(two_share_sd(w, SD_B, SD_S, correlation))
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot([0, 100], [100 * SD_S, 100 * SD_B], color=CORAL, ls="--", lw=1.8, label="the plain average of the two risks")
    ax.plot(100 * ws, 100 * two_share_sd(ws, SD_B, SD_S, correlation), color=SKY, label="the portfolio's standard deviation")
    ax.scatter([bakery_pct], [100 * sd], s=110, color=GOLD, zorder=4)
    ax.set_xlim(0, 100); ax.set_ylim(0, 33); ax.set_xlabel("weight in the bakery (%)")
    ax.set_ylabel("standard deviation (%)"); ax.legend(loc="lower left", frameon=False)
    fig.suptitle(f"{bakery_pct:g} % in the bakery, correlation {correlation:g}: risk {100 * sd:.1f} %, expected return "
                 f"{100 * (w * MEAN_B + (1 - w) * MEAN_S):.2f} %".replace("-", "−"), x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert round(100 * float(two_share_sd(0.75, SD_B, SD_S, RHO)), 1) == 18.4 and float(two_share_sd(0.6, SD_B, SD_S, -1)) < 1e-7

In [ ]:
interact(weight_picture, bakery_pct=FloatSlider(value=50, min=0, max=100, step=5, description="bakery (%)"),
         correlation=FloatSlider(value=0.25, min=-1, max=1, step=0.05, description="correlation"));

### Slide the number of shares

Shares that each have a standard deviation of 40 %, held in equal amounts. With a correlation of 0.25 between every
pair, one share has all 40 points of risk; ten shares have 22.8; a hundred have 20.3. The curve never reaches the
dashed line at 20 %: that floor is the risk the shares have in common, and no amount of spreading removes it. The
shaded part above the floor is specific risk, which washes out. Slide the correlation up, and the floor rises with
it.

In [ ]:
def shares_picture(n=10, correlation=0.25):
    sd_each = 0.40
    avg_var, avg_cov = sd_each ** 2, correlation * sd_each ** 2
    ns = np.arange(1, 101)
    sds, floor_ = equal_weight_sd(ns, avg_var, avg_cov), math.sqrt(avg_cov)
    here = float(equal_weight_sd(n, avg_var, avg_cov))
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.fill_between(ns, 100 * floor_, 100 * sds, color=CHALK, alpha=0.14, label="specific risk: spreading removes it")
    ax.plot(ns, 100 * sds, color=SKY, label="the portfolio's standard deviation")
    ax.axhline(100 * floor_, color=DIM, ls="--", lw=1.8, label="the floor: market risk stays")
    ax.scatter([n], [100 * here], s=110, color=GOLD, zorder=4)
    ax.set_xlim(0, 101); ax.set_ylim(0, 44); ax.set_xlabel("number of shares, in equal amounts")
    ax.set_ylabel("standard deviation (%)"); ax.legend(loc="lower right", frameon=False)
    fig.suptitle(f"{n} share{'s' if n > 1 else ''}: risk {100 * here:.1f} %; the floor is {100 * floor_:.1f} %",
                 x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert [round(100 * float(equal_weight_sd(n, 0.16, 0.04)), 1) for n in (1, 2, 4, 10, 20, 30, 100)] == \
    [40.0, 31.6, 26.5, 22.8, 21.4, 21.0, 20.3]                     # the video's numbers

In [ ]:
interact(shares_picture, n=IntSlider(value=10, min=1, max=100, step=1, description="shares"),
         correlation=FloatSlider(value=0.25, min=0, max=1, step=0.05, description="correlation"));

## Tier 3 · Stretch

Combine ideas, spot the trap, run a formula backwards: the third year a compound average needs, a standard deviation
taken apart, the correlation hidden in a portfolio's risk, the two edges of correlation, how many shares reach a
target, a weight above one, the safest mix of two shares, and a beta turned back into a correlation.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**Running a formula backwards** is the same line of algebra, rearranged. **Searching** is where Python earns its
keep: instead of solving for the safest mix, make an array of every weight in steps of 5 %, compute the risk of all
of them at once, and ask `np.argmin` where the smallest is. Each line computes one stretch answer and checks it.

In [ ]:
third_year = 1.10 ** 3 / (1.30 * 0.80) - 1                         # P3.1: end at 1.1³, standing at 1.3 × 0.8
squares = (10 - 1) * 15 ** 2                                       # P3.2: the standard deviation, taken apart
with_eleventh = math.sqrt(squares / 10)                            #        one more year on the average: divide by 10
hidden = (0.26 ** 2 - 0.25 * 0.20 ** 2 - 0.25 * 0.40 ** 2) / (2 * 0.25 * 0.20 * 0.40)      # P3.3
in_step, opposite = two_share_sd(0.6, 0.20, 0.30, 1.0), two_share_sd(0.6, 0.20, 0.30, -1.0)   # P3.4
enough = math.ceil((0.50 ** 2 - 0.04) / (0.21 ** 2 - 0.04))        # P3.5: 0.21² = 0.04 + (0.25 − 0.04)/n
in_market = (11 - 4) / (9 - 4)                                     # P3.6: above 1, so the extra is borrowed
grid = np.arange(0, 101, 5) / 100                                  # P3.7: every weight in steps of 5 %
risks = two_share_sd(grid, 0.20, 0.30, 0.25)
safest = grid[np.argmin(risks)]
back_to_rho = 0.8 * 20 / 20                                        # P3.8: ρ = β σm/σ
left_over = math.sqrt(20 ** 2 - (0.8 * 20) ** 2)

print(f"the third year must return {third_year:.2%}   squares add up to {squares}; with an eleventh year {with_eleventh:.2f} %")
print(f"the correlation was {hidden:.2f}   in step {in_step:.0%}, opposite {opposite:.0%}   {enough} shares reach 21 %")
print(f"{in_market:g} of your money in the market: {in_market * 20:g} % risk   safest mix {safest:.0%} in the bakery: "
      f"{risks.min():.2%}")
print(f"correlation {back_to_rho:g}, specific risk {left_over:g} %")
for k, v in (("P3.1", [100 * third_year]), ("P3.2", [squares, with_eleventh]), ("P3.3", [hidden]),
             ("P3.4", [100 * in_step, 100 * opposite]), ("P3.5", [enough]), ("P3.6", [in_market, in_market * 20]),
             ("P3.7", [100 * safest, 100 * risks.min()]), ("P3.8", [back_to_rho, left_over])):
    assert agrees(k, v), k

### Slide the mix: which one do you lend and borrow with?

The SKY curve is every mix of the bakery and the software firm: its risk across, its expected return up. The VIOLET
dot is the risk-free rate, 4 %. Pick a mix (the GOLD dot), and the VIOLET line shows everything you can reach by
lending or borrowing alongside it. The slope of that line is the Sharpe ratio: reward for each unit of risk. Slide
until the line is as steep as it can be, just touching the curve: that is 48 % in the bakery, a Sharpe ratio of
0.2251. The safest mix (75 %) is not the best one to lend and borrow with.

In [ ]:
RISK_FREE = 0.04

def line_picture(bakery_pct=50.0):
    w, ws = bakery_pct / 100, np.linspace(0, 1, 201)
    sd, e = float(two_share_sd(w, SD_B, SD_S, RHO)), w * MEAN_B + (1 - w) * MEAN_S
    slope_ = sharpe(e, sd, RISK_FREE)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot(100 * two_share_sd(ws, SD_B, SD_S, RHO), 100 * (ws * MEAN_B + (1 - ws) * MEAN_S), color=SKY,
            label="every mix of the two shares")
    x = np.array([0.0, 40.0])
    ax.plot(x, 100 * RISK_FREE + slope_ * x, color=VIOLET, label="this mix, with lending or borrowing")
    ax.scatter([0], [100 * RISK_FREE], s=90, color=VIOLET, zorder=4, clip_on=False)
    ax.scatter([100 * sd], [100 * e], s=110, color=GOLD, zorder=5)
    ax.set_xlim(0, 36); ax.set_ylim(0, 14); ax.set_xlabel("standard deviation (%)")
    ax.set_ylabel("expected return (%)"); ax.legend(loc="lower right", frameon=False)
    fig.suptitle(f"{bakery_pct:g} % in the bakery: {100 * e:.2f} % at {100 * sd:.1f} % risk; Sharpe ratio {slope_:.4f}",
                 x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

every = np.arange(0, 10001) / 10000                                # the best mix by brute force: every weight to 0.01 %
ratios = sharpe(every * MEAN_B + (1 - every) * MEAN_S, two_share_sd(every, SD_B, SD_S, RHO), RISK_FREE)
assert abs(every[np.argmax(ratios)] - 0.48) < 1e-9 and round(float(ratios.max()), 4) == 0.2251        # the video's numbers
assert round(float(ratios[5000]), 9) == 0.225 and round(float(ratios[7500]), 3) == 0.204

In [ ]:
interact(line_picture, bakery_pct=FloatSlider(value=50, min=0, max=100, step=1, description="bakery (%)"));

## Tier 4 · The case: how risky are Bangalore Cold Chain's shares?

Bangalore Cold Chain runs refrigerated warehouses, and its shares are listed (a fictional company; our own data,
built in code so that the statistics come out round). Over ten years, the shares' yearly returns had a standard
deviation of 40 %, twice the market's 20 %. A director reasons that shares twice as risky as the market deserve
twice the market's risk premium: with a risk-free rate of 4 % and a premium of 5 points, that would be
4 % + 2 × 5 % = 14 %.

**What return do shareholders really ask of the shares?** The next cell holds the ten years of returns on the market
portfolio and on the shares (the same ten rows as the spreadsheet's Case study tab) as a DataFrame, and checks a few
of its numbers.

In [ ]:
PREMIUM = 0.05                                                     # the market's expected return over the risk-free 4 %
# The ten years of returns, as fractions (−0.262 is a loss of 26.2 %), and the case's numbers as the course computed
# them, to check this table against.
# @CASE
MARKET = [
    -0.2622560558842939, 0.08716993924805347, 0.3247663493223254, 0.09766924855891114, 0.17922892782608407,
    -0.2073705460493376, 0.08588212142442599, 0.050887177191081885, 0.3629495278757944, 0.18107331048695524,
]
SHARES = [
    -0.11091030763308968, -0.04704376470547804, 0.0796654686629899, 0.11836851499687817, 0.4606223466426981,
    -0.4900744526816916, -0.5045917061331906, 0.5298333763567388, 0.6443491947460424, 0.3197813297481026,
]
CASE_STATS = {'m_mean': 0.09000000000000001,
 's_mean': 0.09999999999999999,
 'm_sd': 0.20000000000000004,
 's_sd': 0.4,
 'm_var': 0.040000000000000015,
 'cov': 0.04799999999999999,
 'rho': 0.5999999999999998,
 'beta': 1.1999999999999993,
 'ratio': 1.9999999999999998,
 'shortcut': 0.13999999999999999,
 'mkt_part': 0.2399999999999999,
 'spec': 0.3200000000000001,
 'mkt_share': 0.35999999999999965,
 'with10': 0.20649455198624495,
 'with25': 0.22472205054244235,
 'return': 0.09999999999999995,
 'line_sd': 0.2399999999999999,
 'sharpe_alone': 0.14999999999999986,
 'gap': 0.040000000000000036,
 'like_1': 0.4,
 'like_4': 0.28844410203711907,
 'like_16': 0.2529822128134703,
 'like_64': 0.2433105012119287}
# @END CASE
S = CASE_STATS
returns = pd.DataFrame({"year": range(1, 11), "market": MARKET, "shares": SHARES}).set_index("year")
assert returns.shape == (10, 2) and abs(returns["market"].mean() - S["m_mean"]) < 1e-12
assert abs(returns["shares"].std() - S["s_sd"]) < 1e-12 and abs(returns["market"].std() - S["m_sd"]) < 1e-12
print(f"{len(returns)} rows (years 1 to 10), {returns.shape[1]} columns: the same returns as the course's case.")
(100 * returns).round(1)

### How you'd do this in Python

With the returns in a DataFrame, `.mean()` and `.std()` give each column's average and standard deviation, `.cov()`
the table of covariances and `.corr()` the correlations. Beta is one box of the covariance table divided by another:
the shares' covariance with the market, over the market's variance. The cell below reproduces the video's case: the
statistics, the beta two ways, the split of the shares' risk, and the two answers side by side.

In [ ]:
means, sds = returns.mean(), returns.std()                         # .std() divides by n − 1, like STDEV.S
table = returns.cov()                                              # variances on the diagonal, the covariance beside them
cov_sm = table.loc["shares", "market"]                             # COVARIANCE.S
rho_sm = returns.corr().loc["shares", "market"]                    # CORREL
beta_shares = cov_sm / table.loc["market", "market"]               # covariance ÷ the market's variance
beta_again = rho_sm * sds["shares"] / sds["market"]                # correlation × σ ÷ σm: the same number
market_risk = beta_shares * sds["market"]                          # the part that moves with the market
specific_risk = math.sqrt(sds["shares"] ** 2 - market_risk ** 2)   # the part that doesn't
shortcut = RISK_FREE + sds["shares"] / sds["market"] * PREMIUM     # the director: twice the swings, twice the premium
asked = RISK_FREE + beta_shares * PREMIUM                          # the market itself, 1.2 times over

print(f"1 · averages: market {means['market']:.0%}, shares {means['shares']:.0%}   standard deviations: market "
      f"{sds['market']:.0%}, shares {sds['shares']:.0%}")
print(f"2 · covariance {cov_sm:.4f}, correlation {rho_sm:.2f}   beta {beta_shares:.2f} = {cov_sm:.3f}/"
      f"{table.loc['market', 'market']:.2f}, and again {beta_again:.2f} = {rho_sm:.1f} × {sds['shares']:.0%}/{sds['market']:.0%}")
print(f"3 · of the shares' {sds['shares']:.0%}: market risk {market_risk:.0%}, specific risk {specific_risk:.0%} "
      f"({market_risk ** 2 / sds['shares'] ** 2:.0%} of the variance is market risk)")
print(f"4 · the shortcut says {shortcut:.0%}; a mix of the market and borrowing with the same market risk expects {asked:.0%}")
assert abs(cov_sm - S["cov"]) < 1e-12 and abs(rho_sm - S["rho"]) < 1e-12 and abs(beta_shares - S["beta"]) < 1e-12
assert abs(beta_again - S["beta"]) < 1e-12 and abs(market_risk - S["mkt_part"]) < 1e-12 and abs(specific_risk - S["spec"]) < 1e-12
assert abs(shortcut - S["shortcut"]) < 1e-12 and abs(asked - S["return"]) < 1e-12
assert [round(float(v), 4) for v in (cov_sm, rho_sm, beta_shares, market_risk, specific_risk, shortcut, asked)] == \
    [0.048, 0.6, 1.2, 0.24, 0.32, 0.14, 0.10]                      # the video's numbers

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6))
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
x = np.array([-32.0, 42.0])
ax.plot(x, 100 * means["shares"] + 2 * (x - 100 * means["market"]), color=CORAL, ls="--", lw=1.8,
        label="twice the market's moves: the shortcut")
ax.plot(x, 100 * means["shares"] + beta_shares * (x - 100 * means["market"]), color=GOLD,
        label=f"what moves with the market: beta {beta_shares:.1f}")
ax.scatter(100 * returns["market"], 100 * returns["shares"], s=70, color=CHALK, zorder=4, label="one year")
ax.axhline(0, color=DIM, lw=1); ax.axvline(0, color=DIM, lw=1)
ax.set_xlim(-32, 42); ax.set_ylim(-70, 80); ax.set_xlabel("the market's return in the year (%)")
ax.set_ylabel("the shares' return (%)"); ax.legend(loc="upper left", frameon=False)
fig.suptitle(f"Beta {beta_shares:.1f}: only {100 * market_risk:.0f} of the shares' {100 * sds['shares']:.0f} points move with the market; "
             f"the rest scatters around the line", x=0.08, ha="left", fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: move a tenth, then a quarter, of an investor's market portfolio
into the shares; build a portfolio of sixteen companies like this one; and write your own recommendation with the
number that anchors it. Use the DataFrame `returns` and the functions above in a cell of your own (press **+** to
add one), then type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `returns` and checks it against the practice
set: open it when you've finished, or when you're stuck.

In [ ]:
def with_shares(w):
    """An investor in the market portfolio moves a fraction w into the shares: the standard deviation of her
    portfolio's own ten yearly returns."""
    return ((1 - w) * returns["market"] + w * returns["shares"]).std()

def companies_like_it(n):
    """n shares with the same beta, in equal amounts, whose specific risks have nothing to do with each other."""
    return math.sqrt(market_risk ** 2 + specific_risk ** 2 / n)

mine = {}
mine["P4.1"] = [100 * means["market"], 100 * means["shares"], 100 * sds["market"], 100 * sds["shares"]]
mine["P4.2"] = [cov_sm, rho_sm]
mine["P4.3"] = [beta_shares, beta_again]
mine["P4.4"] = [100 * shortcut, 100 * asked]
mine["P4.5"] = [100 * market_risk, 100 * specific_risk, 100 * market_risk ** 2 / sds["shares"] ** 2]
mine["P4.6"] = [100 * with_shares(0.10), 100 * with_shares(0.25)]
mine["P4.7"] = [100 * companies_like_it(16), 100 * market_risk]
mine["P4.8"] = [100 * asked]

for k, got in mine.items():
    got = [float(v) for v in got]
    assert agrees(k, got), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(v, 4))}" for p, v in zip(Q[k]["parts"], got)))
assert abs(with_shares(0.10) - S["with10"]) < 1e-12 and abs(with_shares(0.25) - S["with25"]) < 1e-12
assert all(abs(companies_like_it(n) - S[f"like_{n}"]) < 1e-12 for n in (1, 4, 16, 64))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")
print(f"Shareholders who hold diversified portfolios ask about {asked:.0%} of the shares, not {shortcut:.0%}: only "
      f"{market_risk:.0%} of the {sds['shares']:.0%} moves with the market (beta {beta_shares:.1f}); the other "
      f"{specific_risk:.0%} washes out in a portfolio.")

## Sources
Every exercise, example and company here is our own (Bangalore Cold Chain is fictional), with one exception: the
history of returns is real, public data. Its source is Aswath Damodaran (NYU Stern), "Historical Returns on Stocks,
Bonds and Bills: 1928–2025", the annual returns table at
https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/histretSP.html (retrieved 2026-11-11): three of its
columns (the S&P 500 with dividends, 3-month Treasury bills, 10-year Treasury bonds), and every statistic here is
computed from those 98 rows. It is the record of one country. The order of topics follows the risk and return
chapters of a standard corporate finance course; the formulas are the standard ones for the variance and standard
deviation of a sample, the variance of a portfolio, lending and borrowing at a risk-free rate, the Sharpe ratio and
beta. The spreadsheet functions named in the hints (AVERAGE, GEOMEAN, VAR.S, STDEV.S, COVARIANCE.S, CORREL, SQRT,
SUMPRODUCT) have the same names and take their arguments in the same order in the two most widely used spreadsheets
(their official function lists, retrieved 2026-11-11).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/cf04_risk_portfolios/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("cf04_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert (_q["tier"], _q["group"]) == (_e["tier"], _e["group"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
        _t = _e["tol"] if isinstance(_e["tol"], tuple) else (_e["tol"],) * len(_q["answer"])
        assert list(_t) == _q["tol"], _e["id"]
    assert len(_P.EX) == len(EX)
    _cs = {**{k: v for k, v in _P.CS.items() if k != "like"}, **{f"like_{n}": v for n, v in _P.CS["like"].items()}}
    assert set(_cs) == set(CASE_STATS) and all(abs(float(_cs[k]) - v) < 1e-12 for k, v in CASE_STATS.items())
    assert _P.CASE["market"] == MARKET and _P.CASE["share"] == SHARES and _P.CASE["years"] == list(returns.index)
    assert (_P.CASE["risk_free"], _P.CASE["premium"]) == (RISK_FREE, PREMIUM) == (_P.RF, _P.FACTS["market_premium"])
    assert (_P.YEARS, _P.STOCKS_PCT, _P.BILLS_PCT, _P.BONDS_PCT) == (YEARS, STOCKS, BILLS, BONDS)
    assert _P.BAKERY == bakery and _P.SOFTWARE == software and abs(_P.RHO - RHO) < 1e-12
    assert all(abs(float(history[c].mean()) - _P.H[c]["mean"]) < 1e-12 and abs(float(history[c].std()) - _P.H[c]["sd"]) < 1e-12
               for c in ("stocks", "bills", "bonds")) and abs(risk_premium - _P.H["premium"]) < 1e-12
    assert _P.HISTORY["source"] == "A. Damodaran, NYU Stern · annual returns 1928–2025 · retrieved 2026-11-11"
    print("matches the course's practice set")